##  Shop the look page

In [0]:
%pip install /dbfs/FileStore/shared_uploads/dmat/pmgoogle-1.4-py3-none-any.whl
%pip install  /dbfs/FileStore/shared_uploads/dmat/dmat-0.1-py3-none-any.whl

dbutils.widgets.text(
    "report_run_date",      # Widget name
    "",                     # Default value
    "Report Run Date"       # Optional label
)

In [0]:

from dmat.helpers import *
from dmat import spark_helpers as dsh

from dateutil.relativedelta import relativedelta
import datetime

spark.conf.set("spark.sql.shuffle.partitions","1024")
spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")
#sc.hadoopConfiguration.set("mapreduce.fileoutputcommitter.marksuccessfuljobs", "false")

##############
## Imports & Inits
import pyspark
from pyspark import *
from pyspark.sql import *
from dateutil import tz
from dateutil.relativedelta import relativedelta
import datetime

from pmgoogle.sheets import GoogleSheet
import numpy as np
import pandas as pd
import time
import json
import pyspark.sql.functions as F

# get service account creds
#athena-master@athena-284716.iam.gserviceaccount.com
#service_account = json.loads(dbutils.secrets.get(scope="data_pa-dmat_dmat-secrets-scope", key="athena_sa"))['general']  

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
SaveMode = sc._jvm.org.apache.spark.sql.SaveMode
SaveMode_Append = SaveMode.Append
SaveMode_Overwrite = SaveMode.Overwrite
SaveMode_Ignore = SaveMode.Ignore
SaveMode_Error = SaveMode.ErrorIfExists

## Load default configs for helper libraries and register UDFs
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()


# Returns the current UTC time
def getUTC():
    return datetime.datetime.utcnow().replace(tzinfo=tz.gettz('UTC'))

# Returns the current time in 'US/Pacific'
def getPacificTime(date=False, month=False):
    now = getUTC().astimezone(tz.gettz('US/Pacific'))
    if date:
        now = now.date()
    if month:
        now = now.replace(day=1)

    return now



dsh.init(spark)
Redshift = sc._jvm.com.poshmark.spark.helpers.Redshift

def get_widget(widget_name, default="", _type=str):
    try:
        tmp =  _type(dbutils.widgets.get(widget_name))
        if tmp:
            return tmp
        else:
            return default
    except:
        return default

run_now = get_widget('run_now',default='false')
if (run_now.lower() == 'true'):
    force = True
else:
    force = False

send_day = getPacificTime().isoweekday()
start_of_month = True if getPacificTime().day == 1 else False

## Get today & yesterday
local_date = getPacificTime() 
today = local_date.strftime('%Y-%m-%d')
yesterday = (local_date - datetime.timedelta(days=1)).strftime('%Y-%m-%d')

##############
## get a value from a widget
report_run_date = dbutils.widgets.get("report_run_date")
process_end_date = get_widget('report_run_date', default=yesterday)
## last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=90)).strftime('%Y-%m-%d') 
last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=3)).strftime('%Y-%m-%d')
## last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=2)).strftime('%Y-%m-%d')
process_start_date = last_90_days

## process_date = get_widget("process_date",default=yesterday)
## process_end_date = get_widget("process_end_date",default=yesterday)

# print(f"process_date: {process_date}, process_end_date: {process_end_date}")

# stage_s3_path = "s3://data-tmp-poshmark-production/dmat/project_highway/top_stats/v2/"

def to_jvm_list(py_list):
    l = len(py_list)
    jvm_list = sc._gateway.new_array(sc._jvm.java.lang.String, l)
    for i in range(l):
        jvm_list[i] = py_list[i]
    return jvm_list
print(process_start_date, process_end_date)


In [0]:
from pyspark.sql import *
from pyspark import *
from pyspark.sql import DataFrame

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()

query = "select * from analytics_scratch.l365d_shopper_segment"
DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("l365d_shopper_segment")
# query = "select * from athena_scratch.core__users_active_user_segments_daily"
# DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("core__users_active_user_segments_daily")

val queryDF = spark.sql(""" 
select 
r.event_date,
r.actor.id as user_id,
r.using.app_type as app_type,
dw_users.home_domain, 
CASE WHEN dw_users.gender in ('male','male (hidden)') THEN 'male'
            ELSE 'female' END  AS user_gender,
REPLACE(CAST(dw_users.extended_signup_segments_v3.L06 AS STRING), '"', '') as extended_signup_segments_v3,
daily_au_segment.au_segment  AS daily_au_segment,
coalesce(l365d_shopper_segment_start.shopper_segment_daily,'99. No Purchase')   AS shopper_segment_daily,
at, 
f_pm_epoch_to_pacific(r.at) as event_at,
r.verb,
r.direct_object.name as dob_name,
r.on.name as on_name,
r.properties.story_type as story_type,
r.properties.unit_position as unit_position,
case when r.using.app_type = 'web' then r.properties.content_position - 1 else r.properties.content_position end as new_content_position,
r.properties.content_position as content_position,
r.properties.tracking_meta as tracking_meta,
r.properties.content_type as content_type,
r.properties.location as location, 
r.properties.listing_id as listing_id, 
r.properties.cause, 
get_json_object(properties.tracking_meta, '$.collection_id') as collection_id,

count(case when r.verb = 'view' AND r.direct_object.name = 'look' then r.actor.id else null end) as look_views,
count(case when r.verb = 'observe' AND r.on.name = 'look' then r.actor.id else null end) as look_client_impressions,
-- lookunit client impressions
COUNT(distinct case when r.verb = 'observe' AND r.on.name = 'look' then CONCAT(r.actor.id,f_pm_epoch_to_pacific(r.at),r.properties.story_type,r.properties.unit_position) else null end) as look_unit_impressions,

count(case when r.verb = 'click' AND r.on.name = 'look' AND r.properties.story_type is not null AND r.properties.unit_position is not null then r.actor.id else null end) as look_clicks,

COUNT(CASE WHEN r.verb = 'click' AND r.on.name = 'look'  AND r.properties.unit_position IS NOT NULL AND r.properties.listing_id IS NOT NULL AND r.direct_object.name = 'feed_unit' THEN r.actor.id ELSE NULL END) AS look_listing_clicks
from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
LEFT JOIN external_scratch.core__users_active_user_segments_daily  AS daily_au_segment ON daily_au_segment.user_id = r.actor.id AND (DATE(daily_au_segment.event_date )) = (DATE(r.event_date ))
LEFT JOIN l365d_shopper_segment  AS l365d_shopper_segment_start ON r.actor.id = l365d_shopper_segment_start.buyer_id and

            (DATE(r.event_date )) > (DATE(l365d_shopper_segment_start.start_date )) and  (DATE(r.event_date )) <= (DATE(coalesce(l365d_shopper_segment_start.end_date, current_date())))
WHERE 
((r.verb = 'click' AND r.on.name = 'look' and r.properties.unit_position >= 0)
OR 
(r.verb = 'observe' and r.on.name = 'look' AND r.properties.unit_position >= 0)
OR (r.verb = 'view' and r.direct_object.name = 'look') -- View Event 
)

    AND r.event_date between '$process_end_date' and '$process_start_date'
    --AND r.event_date between '2025-12-08' and '2025-12-08'
    AND r.actor.type IN ('user') 
    AND ((r.using.app_type IN ('iphone', 'ipad', 'android','web')))
    AND dw_users.home_domain in ('us','ca') and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
            coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))

group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
""")
val s3_path = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/ql_visual_discovery_look_page/"
queryDF.write.format("parquet").mode("Overwrite").save(s3_path)

In [0]:
lookDF = spark.sql(f""" 
select 
r.event_date,
r.actor.id as user_id,
r.using.app_type as app_type,
dw_users.home_domain, 
CASE WHEN dw_users.gender in ('male','male (hidden)') THEN 'male'
            ELSE 'female' END  AS user_gender,
REPLACE(CAST(dw_users.extended_signup_segments_v3.L06 AS STRING), '"', '') as extended_signup_segments_v3,
daily_au_segment.au_segment  AS daily_au_segment,
coalesce(l365d_shopper_segment_start.shopper_segment_daily,'99. No Purchase')   AS shopper_segment_daily,
at, 
f_pm_epoch_to_pacific(r.at) as event_at,
r.verb,
r.direct_object.name as dob_name,
r.on.name as on_name,
r.properties.story_type as story_type,
r.properties.unit_position as unit_position,
case when r.using.app_type = 'web' then r.properties.content_position - 1 else r.properties.content_position end as new_content_position,
r.properties.content_position as content_position,
r.properties.tracking_meta as tracking_meta,
r.properties.content_type as content_type,
r.properties.location as location, 
r.properties.listing_id as listing_id, 
r.properties.cause, 
get_json_object(properties.tracking_meta, '$.collection_id') as collection_id,

count(case when r.verb = 'view' AND r.direct_object.name = 'look' then r.actor.id else null end) as look_views,
count(case when r.verb = 'observe' AND r.on.name = 'look' then r.actor.id else null end) as look_client_impressions,
-- lookunit client impressions
COUNT(distinct case when r.verb = 'observe' AND r.on.name = 'look' then CONCAT(r.actor.id,f_pm_epoch_to_pacific(r.at),r.properties.story_type,r.properties.unit_position) else null end) as look_unit_impressions,

count(case when r.verb = 'click' AND r.on.name = 'look' AND r.properties.story_type is not null AND r.properties.unit_position is not null then r.actor.id else null end) as look_clicks,

COUNT(CASE WHEN r.verb = 'click' AND r.on.name = 'look'  AND r.properties.unit_position IS NOT NULL AND r.properties.listing_id IS NOT NULL AND r.direct_object.name = 'feed_unit' THEN r.actor.id ELSE NULL END) AS look_listing_clicks
from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
LEFT JOIN external_scratch.core__users_active_user_segments_daily  AS daily_au_segment ON daily_au_segment.user_id = r.actor.id AND (DATE(daily_au_segment.event_date )) = (DATE(r.event_date ))
LEFT JOIN l365d_shopper_segment  AS l365d_shopper_segment_start ON r.actor.id = l365d_shopper_segment_start.buyer_id and

            (DATE(r.event_date )) > (DATE(l365d_shopper_segment_start.start_date )) and  (DATE(r.event_date )) <= (DATE(coalesce(l365d_shopper_segment_start.end_date, current_date())))
WHERE 
((r.verb = 'click' AND r.on.name = 'look' and r.properties.unit_position >= 0)
OR 
(r.verb = 'observe' and r.on.name = 'look' AND r.properties.unit_position >= 0)
OR (r.verb = 'view' and r.direct_object.name = 'look') -- View Event 
)

    --AND r.event_date between '$process_end_date' and '$process_start_date'
    AND r.event_date between '{process_start_date}' and '{process_end_date}'
    AND r.actor.type IN ('user') 
    AND ((r.using.app_type IN ('iphone', 'ipad', 'android','web')))
    AND dw_users.home_domain in ('us','ca') and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
            coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))

group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
""")


#### below method is directly write to S3 for history

```python move to 1 year bucket```
## bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/look"
data_name = "lookDF"

output_path_old = f"{bucket}/{team_name}/{project_name}/{data_name}/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/product_analytics/feed/visual_discovery/look/lookDF/"
df = spark.read.format("parquet").load(output_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:
## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time

bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
#bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/look"
data_name = "lookDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

## ## temp change to static from dynamic
##spark.conf.set("spark.sql.sources.partitionOverwriteMode","static")

lookDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_visual_discovery_look_page")

##print(process_start_date)
##print(process_end_date)
print(output_path)

In [0]:
# #bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
# bucket = "s3://analytics-scratch-reviewed-poshmark-production" 
# team_name = "product_analytics"
# project_name = "feed/visual_discovery/look"
# data_name = "lookDF"

# output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"
# look_sdf = spark.read.parquet(output_path)
# look_sdf.createOrReplaceTempView("ql_visual_discovery_look_page")

In [0]:
# %sql 
# select min(event_date), max(event_date) from ql_visual_discovery_look_page

In [0]:
# %sql 
# select * from ql_visual_discovery_look_page
# limit 5

### Look CTR Details

In [0]:
# %sql

# select 
# event_date, 
# app_type, 
# user_gender, 
# home_domain, 
# daily_au_segment,
# shopper_segment_daily, 
# unit_position, 
# content_position, 
# location, 
# COUNT(*),
# COUNT(case when r.verb = 'view' AND r.dob_name = 'look' then r.user_Id else null end) as look_views, 
# COUNT(case when r.verb = 'observe' AND r.on_name = 'look' then r.user_Id else null end) as look_client_impressions,
# -- lookunit client impressions
# COUNT(distinct case when r.verb = 'observe' AND r.on_name = 'look' then CONCAT(r.user_Id,f_pm_epoch_to_pacific(r.at),r.story_type,r.unit_position) else null end) as look_unit_impressions,
# /** clicks**/
# COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit' 
#         and content_type = 'post' then r.user_Id else null end) as look_clicks,
        
# COUNT(CASE WHEN r.verb = 'click' AND r.on_name = 'look'  AND r.unit_position IS NOT NULL AND r.listing_id IS NOT NULL AND r.dob_name = 'feed_unit'  and content_type = 'post'  THEN r.user_Id ELSE NULL END) AS look_listing_clicks, --listing_ID IS NOT NULL

# COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post'  and r.location = 'content' then r.user_Id else null end) as look_content_clicks,
# COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post' and r.location = 'footer' then user_Id else null end) as look_footer_clicks,

# /** clickers**/

# COUNT(DISTINCT case when r.verb = 'view' AND r.dob_name = 'look' then r.user_Id else null end) as look_viewers,
# COUNT(DISTINCT case when r.verb = 'observe' AND r.on_name = 'look' then r.user_Id else null end) as look_client_impressions_user,

# COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit' 
#         and content_type = 'post' then r.user_Id else null end) as look_clickers,
# COUNT(DISTINCT CASE WHEN r.verb = 'click' AND r.on_name = 'look'  AND r.unit_position IS NOT NULL AND r.listing_id IS NOT NULL AND r.dob_name = 'feed_unit'  and content_type = 'post'  THEN r.user_Id ELSE NULL END) AS look_listing_clicker, --listing_ID IS NOT NULL

# COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post'  and r.location = 'content' then r.user_Id else null end) as look_content_clicker,
# COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post' and r.location = 'footer' then user_Id else null end) as look_footer_clickers
# from ql_visual_discovery_look_page r
# where event_date between  '2026-01-09' and '2026-01-28'
# group by 1,2,3,4,5,6,7,8,9



In [0]:
# s3_path = "s3://data-tmp-poshmark-production/harshal/feed_personalisation/daily_user_level_preference_data/daily_user_level_preference_data_raw"
# df = spark.read.format("parquet").load(s3_path)
# df.createOrReplaceTempView("daily_user_level_preference_data_raw")

In [0]:
# %sql 
# select * from daily_user_level_preference_data_raw 
# limit 5

added user preference on 1/29

In [0]:
# Adding User preferences 

In [0]:
# queryDF = spark.sql(f"""

# with cte as (
# select * from daily_user_level_preference_data_raw
# where event_date between '2026-02-23' and '2026-03-08'
# ) 
# select 
# r.event_date, 
# r.app_type, 
# r.user_gender, 
# r.home_domain, 
# r.daily_au_segment,
# r.shopper_segment_daily, 
# dw_users.generation, 

# case when r.extended_signup_segments_v3 
# in ('001', '006', '012', '019', '028', '040', '064', '066', '080', '082', '092', '099', '100', '102', '107', '124'
# ) then 'Test_NonPersonalized'
# when r.extended_signup_segments_v3 
# in ('002', '014', '015', '023', '024', '035', '046', '057', '063', '067', '070', '104', '112', '119', '125', '127'
# ) then 'Test_Personalized'
# when r.extended_signup_segments_v3 
# in ('003', '009', '016', '017', '018', '021', '030', '032', '051', '054', '073', '074', '075', '076', '078', '105') then 'Control' 
# else 'Other'
# end as exp_grp, 

# CASE 
# WHEN total_preferences IS NULL OR total_preferences = 0 THEN '1. 0' 
# WHEN total_preferences BETWEEN 1 AND 3 THEN '2. 1-3' 
# WHEN total_preferences BETWEEN 4 AND 6 THEN '3. 4-6' 
# WHEN total_preferences BETWEEN 7 AND 10 THEN '4. 7-10' 
# WHEN total_preferences BETWEEN 11 AND 20 THEN '5. 11-20' 
# WHEN total_preferences BETWEEN 21 AND 30 THEN '6. 21-30' 
# WHEN total_preferences BETWEEN 31 AND 50 THEN '7. 31-50' 
# WHEN total_preferences BETWEEN 51 AND 100 THEN '8. 51-100' 
# WHEN total_preferences >= 101 THEN '9. 100+' ELSE '9.1. other' END AS total_preferences_bucket,
# CASE
#     WHEN brand_preferences IS NULL OR brand_preferences = 0 THEN '1. 0'
#     WHEN brand_preferences BETWEEN 1 AND 5 THEN '2. 1-5'
#     WHEN brand_preferences BETWEEN 6 AND 10 THEN '3. 6-10'
#     WHEN brand_preferences BETWEEN 11 AND 20 THEN '4. 11-20'
#     WHEN brand_preferences BETWEEN 21 AND 50 THEN '5. 21-50'
#     WHEN brand_preferences BETWEEN 51 AND 100 THEN '6. 51-100'
#     WHEN brand_preferences BETWEEN 101 AND 200 THEN '7. 101-200'
#     WHEN brand_preferences > 200 THEN '8. 200+'
#     ELSE '9. other'
# END AS brand_pref_bucket,
# CASE
#     WHEN aesthetic_preferences IS NULL OR aesthetic_preferences = 0 THEN '1. 0'
#     WHEN aesthetic_preferences BETWEEN 1 AND 5 THEN '2. 1-5'
#     WHEN aesthetic_preferences BETWEEN 6 AND 10 THEN '3. 6-10'
#     WHEN aesthetic_preferences BETWEEN 11 AND 20 THEN '4. 11-20'
#     WHEN aesthetic_preferences BETWEEN 21 AND 50 THEN '5. 21-50'
#     WHEN aesthetic_preferences BETWEEN 51 AND 100 THEN '6. 51-100'
#     WHEN aesthetic_preferences BETWEEN 101 AND 200 THEN '7. 101-200'
#     WHEN aesthetic_preferences > 200 THEN '8. 200+'
#     ELSE '9. other'
# END AS aesthetic_pref_bucket, 
# unit_position, 
# content_position, 
# location, 
# COUNT(*),
# COUNT(case when r.verb = 'view' AND r.dob_name = 'look' then r.user_Id else null end) as look_views, 
# COUNT(case when r.verb = 'observe' AND r.on_name = 'look' then r.user_Id else null end) as look_client_impressions,
# -- lookunit client impressions
# COUNT(distinct case when r.verb = 'observe' AND r.on_name = 'look' then CONCAT(r.user_Id,f_pm_epoch_to_pacific(r.at),r.story_type,r.unit_position) else null end) as look_unit_impressions,
# /** clicks**/
# COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit' 
#         and content_type = 'post' then r.user_Id else null end) as look_clicks,
        
# COUNT(CASE WHEN r.verb = 'click' AND r.on_name = 'look'  AND r.unit_position IS NOT NULL AND r.listing_id IS NOT NULL AND r.dob_name = 'feed_unit'  and content_type = 'post'  THEN r.user_Id ELSE NULL END) AS look_listing_clicks, --listing_ID IS NOT NULL

# COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post'  and r.location = 'content' then r.user_Id else null end) as look_content_clicks,
# COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post' and r.location = 'footer' then r.user_Id else null end) as look_footer_clicks,

# /** clickers**/

# COUNT(DISTINCT case when r.verb = 'view' AND r.dob_name = 'look' then r.user_Id else null end) as look_viewers,
# COUNT(DISTINCT case when r.verb = 'observe' AND r.on_name = 'look' then r.user_Id else null end) as look_client_impressions_user,

# COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit' 
#         and content_type = 'post' then r.user_Id else null end) as look_clickers,
# COUNT(DISTINCT CASE WHEN r.verb = 'click' AND r.on_name = 'look'  AND r.unit_position IS NOT NULL AND r.listing_id IS NOT NULL AND r.dob_name = 'feed_unit'  and content_type = 'post'  THEN r.user_Id ELSE NULL END) AS look_listing_clicker, --listing_ID IS NOT NULL

# COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post'  and r.location = 'content' then r.user_Id else null end) as look_content_clicker,
# COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post' and r.location = 'footer' then r.user_Id else null end) as look_footer_clickers
# from ql_visual_discovery_look_page r
# left join  cte l  on r.user_id = l.user_id and r.event_Date = l.event_date
# inner join s3_analytics.dw_users  AS dw_users ON r.user_id = dw_users.user_id
# where r.event_date between  '2026-02-23' and '2026-03-08' 
# --and dw_users.gender in  ('female', 'female (hidden)') and dw_users.home_domain = 'us'
# group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14
# """
# )
# s3_path = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/ql_visual_discovery_look_page_summary2/"
# queryDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)



In [0]:
# queryDF.toPandas().head()

In [0]:
# s3_path =  "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/ql_visual_discovery_look_page_summary2/"
# look_sdf = spark.read.parquet(s3_path)
# look_sdf.createOrReplaceTempView("ql_visual_discovery_look_page_summary")


In [0]:
# %sql 
# select user_gender, 
# count(*) from ql_visual_discovery_look_page_summary 
# group by 1

In [0]:
# %sql 
# select daily_au_segment, 
# shopper_segment_daily, 
# generation, 
# exp_grp,
# total_preferences_bucket, 
# brand_pref_bucket, 
# aesthetic_pref_bucket, 
# sum(look_client_impressions)  look_client_impressions, 
# sum(look_clicks)  look_clicks, 
# sum(look_listing_clicks) look_listing_clicks, 
# sum(look_footer_clicks)look_footer_clicks
# from ql_visual_discovery_look_page_summary 
# where home_domain = 'us' and user_gender = 'female'  and app_type = 'iphone'
# and content_position <=3 and unit_position >0
# group by 1,2,3,4,5,6,7

In [0]:
# %sql
# select 
# event_date,
  
# COUNT(case when r.verb = 'view' AND r.dob_name = 'look' then r.user_Id else null end) as look_views, 
# COUNT(case when r.verb = 'observe' AND r.on_name = 'look' then r.user_Id else null end) as look_client_impressions,

# -- lookunit client impressions
# COUNT(distinct case when r.verb = 'observe' AND r.on_name = 'look' then CONCAT(r.user_Id,f_pm_epoch_to_pacific(r.at),r.story_type,r.unit_position) else null end) as look_unit_impressions,

# /** clicks**/
# COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit' 
#         and content_type = 'post' then r.user_Id else null end) as look_clicks,
        
# COUNT(CASE WHEN r.verb = 'click' AND r.on_name = 'look'  AND r.unit_position IS NOT NULL AND r.listing_id IS NOT NULL AND r.dob_name = 'feed_unit'  and content_type = 'post'  THEN r.user_Id ELSE NULL END) AS look_listing_clicks, --listing_ID IS NOT NULL

# COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post'  and r.location = 'content' then r.user_Id else null end) as look_content_clicks,
# COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post' and r.location = 'footer' then user_Id else null end) as look_footer_clicks,

# /** clickers**/

# COUNT(DISTINCT case when r.verb = 'view' AND r.dob_name = 'look' then r.user_Id else null end) as look_viewers,
# COUNT(DISTINCT case when r.verb = 'observe' AND r.on_name = 'look' then r.user_Id else null end) as look_client_impressions_user,

# COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit' 
#         and content_type = 'post' then r.user_Id else null end) as look_clickers,
# COUNT(DISTINCT CASE WHEN r.verb = 'click' AND r.on_name = 'look'  AND r.unit_position IS NOT NULL AND r.listing_id IS NOT NULL AND r.dob_name = 'feed_unit'  and content_type = 'post'  THEN r.user_Id ELSE NULL END) AS look_listing_clicker, --listing_ID IS NOT NULL

# COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post'  and r.location = 'content' then r.user_Id else null end) as look_content_clicker,
# COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post' and r.location = 'footer' then user_Id else null end) as look_footer_clickers
# from ql_visual_discovery_look_page r
# --where app_type <>'web' and ((r.content_position <=4 and r.verb in ( 'click','observe') )or r.verb = 'view')--removed content position >4 (tracking issue)
# where event_date >= '2026-01-09'
# group by 1
# order by 1 



In [0]:
# %md
# ## 2. 0 Investigation 
# #### 1) web missing listing ID; 


In [0]:
# %sql 
# select app_type, story_type, content_type,location, count(*), count(case when listing_id is null then 1 else null end ) as list_id_missing_cnt
# from ql_visual_discovery_look_page 
# where 1=1 and verb = 'click' and location in ( 'content','footer') 
# and event_date >= '2026-01-09'
# group by 1,2,3,4


#### 2) click events without observe events 

In [0]:
# %sql 

# select *
# from ql_visual_discovery_look_page 
# where 1=1 and verb = 'click' and content_position >=4  and app_type <>'web'
# and event_date = '2025-12-07'
# limit 5


In [0]:
# %sql 

# select user_id, sum(look_client_impressions) look_client_impressions,  sum(look_clicks)look_clicks
# from ql_visual_discovery_look_page 
# where 1=1  and content_position >=4  and app_type <>'web'
# and event_date = '2025-12-07'
# group by 1 
# having look_clicks >look_client_impressions
# order by look_clicks desc 
# limit 10